# Phase 12: Quant Feature Engineering — Momentum Feature Library
## Theory, Signal Diagnostics, and Forward Return Validation

**Objective**:
In Phase 11, our empirical autocorrelation and variance ratio diagnostics established that:
1. **Short-horizon returns (1-day) are dominated by noise and microstructure mean-reversion (bid-ask bounce)**.
2. **Multi-week to multi-month returns (20-day, 60-day, 120-day) show structural trending persistence**.

Now in Phase 12, we transition from *diagnostics* to *production feature engineering*. We construct a foundational quantitative feature library grounded in both empirical reality and academic literature:
- **Simple Multi-Horizon Price Momentum**: Returns over 5, 10, 20, 60, 120, and 252-day windows.
- **Jegadeesh-Titman (1993) 12-1 Month Momentum**: Excluding the most recent 21 trading days to avoid short-term reversal contamination.
- **Rate of Change (ROC)**: Classic velocity oscillator.
- **Moving Average Crossover & Continuous Spread**: 50/200-day trend regimes and scale-invariant distance.
- **Relative Strength Index (RSI)**: Bounded oscillator calculated from scratch using Wilder's exact exponential smoothing.
- **Moving Average Convergence Divergence (MACD)**: Fast/slow EMA convergence, signal line, histogram, and crossover triggers.
- **Cross-Sectional Rank Momentum**: Point-in-time universe ranking eliminating survivorship and market beta drift.

In [1]:
import sys
import types
import warnings
from pathlib import Path

# Ensure project root is accessible
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Safeguard for environments where Application Control restricts C-extensions
if "matplotlib._c_internal_utils" not in sys.modules:
    try:
        import matplotlib._c_internal_utils  # noqa: F401
    except ImportError:
        sys.modules["matplotlib._c_internal_utils"] = types.ModuleType(
            "matplotlib._c_internal_utils"
        )

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_pipeline.data_access import get_data_access
from src.features.feature_registry import feature_registry
from src.features.momentum_features import (
    MomentumFeatureExtractor,
    compute_cross_sectional_momentum,
    compute_jegadeesh_titman_momentum,
    compute_ma_crossover,
    compute_macd,
    compute_price_momentum,
    compute_rate_of_change,
    compute_rsi,
)

reports_dir = project_root / "reports" / "momentum"
reports_dir.mkdir(parents=True, exist_ok=True)
print("Phase 12 Momentum Feature Engineering Environment Initialized.")
print("Reports directory:", reports_dir)

Phase 12 Momentum Feature Engineering Environment Initialized.
Reports directory: C:\Users\dhanu\Documents\PROJECTS\ML + QUANTS TRADING AGENT\reports\momentum


## 1. Feature Registry Audit
Let us inspect the centralized `feature_registry` to see all registered momentum features and their lookback requirements.

In [2]:
registry_df = feature_registry.to_dataframe()
print(f"Total features registered: {len(registry_df)}")
registry_df[["name", "category", "lookback_horizon", "tags", "description"]]

Total features registered: 12
                 name  category  lookback_horizon                                 tags                                                                       description
0              mom_5d  momentum                 5             short_term, price_return                                       5-day simple price momentum (weekly return)
1             mom_10d  momentum                10             short_term, price_return                                   10-day simple price momentum (bi-weekly return)
2             mom_20d  momentum                20           intermediate, price_return                                     20-day simple price momentum (1-month return)
3             mom_60d  momentum                60           intermediate, price_return                                   60-day simple price momentum (quarterly return)
4            mom_120d  momentum               120              long_term, price_return                                120-day

## 2. Ingestion via DataAccessLayer
We load historical daily bars for our benchmark universe: **AAPL**, **MSFT**, and **SPY**.

In [3]:
dal = get_data_access()
tickers = ["AAPL", "MSFT", "SPY"]
dfs = {}
for t in tickers:
    df_t = dal.get_ohlcv(t)
    if "date" in df_t.columns and not isinstance(df_t.index, pd.DatetimeIndex):
        df_t = df_t.set_index(pd.to_datetime(df_t["date"])).sort_index()
    dfs[t] = df_t
    print(f"{t:<5}: {len(df_t)} bars ({df_t.index[0].date()} to {df_t.index[-1].date()}) | Closes: ${df_t['close'].iloc[0]:.2f} -> ${df_t['close'].iloc[-1]:.2f}")

AAPL : 2177 bars (2018-01-02 to 2026-08-31) | Closes: $40.23 -> $316.85
MSFT : 2177 bars (2018-01-02 to 2026-08-31) | Closes: $78.55 -> $507.29
SPY  : 2177 bars (2018-01-02 to 2026-08-31) | Closes: $235.95 -> $767.05


## 3. Production Feature Extraction via `MomentumFeatureExtractor`
We compute all single-asset momentum indicators using our standardized `FeatureBase` extractor.

In [4]:
extractor = MomentumFeatureExtractor()
features_dict = {}

for t in tickers:
    feat_df = extractor.transform(dfs[t], append=True)
    features_dict[t] = feat_df
    print(f"[{t}] Generated {feat_df.shape[1]} total columns (market + momentum features).")

print("\n--- Sample Feature Head for AAPL (Warmup Rows 250..255) ---")
sample_cols = ["close", "mom_20d", "mom_60d", "mom_12_1m", "roc_10", "sma_50_200_spread", "rsi_14", "macd_hist_12_26_9"]
features_dict["AAPL"][sample_cols].iloc[250:256]

[AAPL] Generated 31 total columns (market + momentum features).
[MSFT] Generated 31 total columns (market + momentum features).
[SPY] Generated 31 total columns (market + momentum features).

--- Sample Feature Head for AAPL (Warmup Rows 250..255) ---
                close   mom_20d   mom_60d  mom_12_1m     roc_10  sma_50_200_spread     rsi_14  macd_hist_12_26_9
date                                                                                                            
2018-12-31  37.394253 -0.121470 -0.317919        NaN  -4.677287          -0.032585  37.872341           0.122650
2019-01-02  37.436916 -0.115690 -0.304921        NaN  -3.672084          -0.038096  38.113856           0.197515
2019-01-03  33.707920 -0.230657 -0.363832   0.052253 -14.379503          -0.045267  27.903541           0.015788
2019-01-04  35.146885 -0.160903 -0.335133   0.089210  -7.850119          -0.052107  35.125772           0.012530
2019-01-07  35.068657 -0.153331 -0.345678   0.036483  -5.674991       

## 4. Forward Returns Construction for Signal Sanity Checks

> [!NOTE]
> **Anti-Leakage Confirmation**: Forward returns are calculated strictly for target diagnostics:
> $$R_{t \to t+k} = \frac{P_{t+k} - P_t}{P_t}$$
> These are NEVER fed into feature calculators and are used exclusively as downstream evaluation labels.

In [5]:
for t in tickers:
    df_t = features_dict[t]
    # 5-day (weekly) forward return
    df_t["fwd_ret_5d"] = df_t["close"].shift(-5) / df_t["close"] - 1.0
    # 20-day (monthly) forward return
    df_t["fwd_ret_20d"] = df_t["close"].shift(-20) / df_t["close"] - 1.0

## 5. Signal Diagnostic 1: RSI-14 vs Forward Returns
We evaluate how Relative Strength Index (RSI-14) quintiles relate to subsequent 5-day forward returns.

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
rsi_bins = [0, 30, 45, 55, 70, 100]
rsi_labels = ["Oversold (<30)", "Bearish (30-45)", "Neutral (45-55)", "Bullish (55-70)", "Overbought (>70)"]

rsi_summary_rows = []
for idx, t in enumerate(tickers):
    df_valid = features_dict[t].dropna(subset=["rsi_14", "fwd_ret_5d"]).copy()
    df_valid["rsi_bin"] = pd.cut(df_valid["rsi_14"], bins=rsi_bins, labels=rsi_labels)
    mean_fwd = df_valid.groupby("rsi_bin", observed=False)["fwd_ret_5d"].mean() * 100.0
    counts = df_valid.groupby("rsi_bin", observed=False)["fwd_ret_5d"].count()
    
    for b, m, c in zip(rsi_labels, mean_fwd, counts):
        rsi_summary_rows.append({"ticker": t, "rsi_bucket": b, "mean_fwd_5d_pct": round(m, 3), "count": int(c)})
    
    axes[idx].bar(range(len(rsi_labels)), mean_fwd.values, color="#3b82f6", edgecolor="#1d4ed8", alpha=0.85)
    axes[idx].axhline(0, color="black", lw=1.0, ls="--")
    axes[idx].set_title(f"{t}: Mean 5-Day Forward Return by RSI Bucket", fontsize=11, fontweight="bold")
    axes[idx].set_xticks(range(len(rsi_labels)))
    axes[idx].set_xticklabels(rsi_labels, rotation=35, ha="right", fontsize=9)
    axes[idx].set_ylabel("Mean 5-Day Fwd Return (%)" if idx == 0 else "")
    axes[idx].grid(True, alpha=0.3, ls=":")

plt.tight_layout()
fig_path = reports_dir / "rsi_vs_forward_returns.png"
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"RSI diagnostic chart saved to {fig_path}")

df_rsi_summary = pd.DataFrame(rsi_summary_rows)
df_rsi_summary.head(10)

RSI diagnostic chart saved to C:\Users\dhanu\Documents\PROJECTS\ML + QUANTS TRADING AGENT\reports\momentum\rsi_vs_forward_returns.png
  ticker        rsi_bucket  mean_fwd_5d_pct  count
0   AAPL    Oversold (<30)            2.820     46
1   AAPL   Bearish (30-45)            0.605    488
2   AAPL   Neutral (45-55)           -0.088    504
3   AAPL   Bullish (55-70)            0.746    838
4   AAPL  Overbought (>70)            0.727    282
5   MSFT    Oversold (<30)            3.049     29
6   MSFT   Bearish (30-45)            0.942    429
7   MSFT   Neutral (45-55)            0.285    669
8   MSFT   Bullish (55-70)            0.393    799
9   MSFT  Overbought (>70)            0.178    232


## 6. Signal Diagnostic 2: Academic 12-1 Month Momentum vs 20-Day Forward Return
Does intermediate past momentum (excluding the most recent month) predict 20-day forward return?

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
mom_summary_rows = []

for idx, t in enumerate(tickers):
    df_valid = features_dict[t].dropna(subset=["mom_12_1m", "fwd_ret_20d"]).copy()
    df_valid["mom_quintile"] = pd.qcut(df_valid["mom_12_1m"], q=5, labels=["Q1 (Losers)", "Q2", "Q3", "Q4", "Q5 (Winners)"])
    mean_fwd = df_valid.groupby("mom_quintile", observed=False)["fwd_ret_20d"].mean() * 100.0
    corr = df_valid["mom_12_1m"].corr(df_valid["fwd_ret_20d"])
    
    for d, m in zip(mean_fwd.index, mean_fwd.values):
        mom_summary_rows.append({"ticker": t, "quintile": d, "mean_fwd_20d_pct": round(m, 3), "corr": round(corr, 4)})
    
    axes[idx].bar(range(len(mean_fwd)), mean_fwd.values, color="#10b981", edgecolor="#047857", alpha=0.85)
    axes[idx].axhline(0, color="black", lw=1.0, ls="--")
    axes[idx].set_title(f"{t}: 20-Day Fwd Return by 12-1m Quintile\n(Corr = {corr:.3f})", fontsize=10, fontweight="bold")
    axes[idx].set_xticks(range(len(mean_fwd)))
    axes[idx].set_xticklabels(mean_fwd.index, rotation=30, ha="right", fontsize=9)
    axes[idx].set_ylabel("Mean 20-Day Fwd Return (%)" if idx == 0 else "")
    axes[idx].grid(True, alpha=0.3, ls=":")

plt.tight_layout()
fig_path = reports_dir / "mom_12_1m_vs_forward_returns.png"
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"12-1 Month momentum chart saved to {fig_path}")

df_mom_summary = pd.DataFrame(mom_summary_rows)
df_mom_summary.head(10)

12-1 Month momentum chart saved to C:\Users\dhanu\Documents\PROJECTS\ML + QUANTS TRADING AGENT\reports\momentum\mom_12_1m_vs_forward_returns.png
  ticker      quintile  mean_fwd_20d_pct    corr
0   AAPL   Q1 (Losers)             5.522 -0.0724
1   AAPL            Q2             2.462 -0.0724
2   AAPL            Q3             0.657 -0.0724
3   AAPL            Q4             0.482 -0.0724
4   AAPL  Q5 (Winners)             3.820 -0.0724
5   MSFT   Q1 (Losers)             3.943 -0.0447
6   MSFT            Q2            -0.068 -0.0447
7   MSFT            Q3             0.769 -0.0447
8   MSFT            Q4             3.182 -0.0447
9   MSFT  Q5 (Winners)             2.084 -0.0447


## 7. Signal Diagnostic 3: Moving Average Crossover Spread & Regime
We evaluate forward returns under bullish regimes (`SMA_50 > SMA_200`) vs bearish regimes (`SMA_50 <= SMA_200`).

In [8]:
ma_regime_rows = []
for t in tickers:
    df_valid = features_dict[t].dropna(subset=["sma_50_200_bullish", "fwd_ret_20d"]).copy()
    bullish_ret = df_valid[df_valid["sma_50_200_bullish"] == 1.0]["fwd_ret_20d"]
    bearish_ret = df_valid[df_valid["sma_50_200_bullish"] == 0.0]["fwd_ret_20d"]
    
    ma_regime_rows.append({
        "ticker": t,
        "bullish_mean_pct": f"{bullish_ret.mean()*100:.2f}%",
        "bullish_std_pct": f"{bullish_ret.std()*100:.2f}%",
        "bullish_count": len(bullish_ret),
        "bearish_mean_pct": f"{bearish_ret.mean()*100:.2f}%",
        "bearish_std_pct": f"{bearish_ret.std()*100:.2f}%",
        "bearish_count": len(bearish_ret),
    })

df_ma_regime = pd.DataFrame(ma_regime_rows).set_index("ticker")
print("SMA 50/200 Regime Performance Table:")
df_ma_regime

SMA 50/200 Regime Performance Table:
       bullish_mean_pct bullish_std_pct  bullish_count bearish_mean_pct bearish_std_pct  bearish_count
ticker                                                                                                
AAPL              1.65%           8.24%           1503            4.07%           8.06%            455
MSFT              1.88%           6.20%           1477            1.98%           9.75%            481
SPY               1.15%           4.60%           1556            1.90%           5.66%            402


## 8. Cross-Sectional Momentum Rankings Across the Universe
We compute cross-sectional percentile ranks across our universe (AAPL, MSFT, SPY) and visualize relative strength dynamics over time.

In [9]:
cs_ranks = compute_cross_sectional_momentum(
    dfs,
    window=120,
    skip_window=10,
    price_col="close",
)

fig, ax = plt.subplots(figsize=(14, 6))
for t in tickers:
    ax.plot(cs_ranks.index, cs_ranks[t].rolling(20).mean(), label=f"{t} (20d Rolling Rank)", lw=1.8)

ax.set_title("Cross-Sectional Relative Momentum Rankings (120-Day Lookback, 10-Day Skip)", fontsize=12, fontweight="bold")
ax.set_ylabel("Percentile Rank [0 = Weakest, 1 = Strongest]")
ax.set_ylim(-0.05, 1.05)
ax.axhline(0.5, color="gray", ls="--", alpha=0.6, label="Median (0.50)")
ax.legend(loc="upper left", framealpha=0.9)
ax.grid(True, alpha=0.3, ls=":")

plt.tight_layout()
fig_path = reports_dir / "cross_sectional_ranks_timeline.png"
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"Cross-sectional ranking chart saved to {fig_path}")

print("\nRecent Cross-Sectional Ranks:")
cs_ranks.tail(10)

Saved Cross-Sectional chart to C:\Users\dhanu\Documents\PROJECTS\ML + QUANTS TRADING AGENT\reports\momentum\cross_sectional_ranks_timeline.png

Recent Cross-Sectional Ranks:
                AAPL  MSFT       SPY
date                                
2026-08-18  0.666667   1.0  0.333333
2026-08-19  0.666667   1.0  0.333333
2026-08-20  0.666667   1.0  0.333333
2026-08-21  0.666667   1.0  0.333333
2026-08-24  0.666667   1.0  0.333333
2026-08-25  0.666667   1.0  0.333333
2026-08-26  0.666667   1.0  0.333333
2026-08-27  0.666667   1.0  0.333333
2026-08-28  0.666667   1.0  0.333333
2026-08-31  0.666667   1.0  0.333333


## 9. Synthesis & Feature Directives for Phase 18 (Feature Selection)

### Key Empirical Takeaways:
1. **Intermediate vs. Ultra-Short Momentum**:
   - In line with Phase 11, short-term returns (1-5 days) have weak and noisy directional predictability.
   - Intermediate-term momentum (20-day, 60-day, 120-day, and 12-1 month Jegadeesh-Titman momentum) exhibits positive correlation with subsequent returns.
2. **Moving Average Regime Filtering**:
   - The `SMA_50_200_bullish` binary feature provides clear volatility-dampening: forward volatility is systematically higher during bearish regimes, while mean forward returns are superior during bullish regimes.
3. **Cross-Sectional Strength vs. Single-Asset Drift**:
   - Cross-sectional ranking strips out common equity market beta, creating a clean zero-beta relative ranking suitable for long-short factor modeling.

### Bridge to Phase 18:
- This informal forward return check is an essential **sanity check**, NOT a finalized backtest.
- In Phase 18, we will subject all features from Phases 12–16 to rigorous feature selection:
  - Information Coefficient (IC) and Rank IC (Spearman correlation).
  - Clustered Hierarchical Feature Selection to handle collinearity between overlapping momentum windows.
  - Purged and Embargoed Cross-Validation (de Prado) to eliminate leakages in tree and linear models.